# Custom UNet: build it from config and train it with the library

*Tutorial for issue #61 (map #12) — nothing here is exported to `src/` (no `#| export`, not run by `nbdev-test`). CPU-only, synthetic data.*

Until now `get_model()` always built an **SMP** model: a pretrained encoder plus a decoder from `segmentation-models-pytorch`.
Since #61 you can ask for a **library-owned classic UNet** instead (double-conv blocks, GroupNorm, plain encoder/decoder).
It is switched on with one config key:

```yaml
model:
  source: custom_unet        # smp (default) | custom_unet
  custom_unet:
    encoder_depth: 4         # how many times the image is halved (integer for now)
    base_channels: 32        # channels in the first stage; doubles each stage up to max_channels
    max_channels: 512
    num_groups: 8            # GroupNorm groups (shrunk automatically if they do not divide the channels)
```

What we check, in order:
1. The factory returns the right model for each `model.source`, and the output shape is `(B, classes, H, W)`.
2. Why training treats the custom model differently from a pretrained one (it starts from random weights).
3. A real `train()` run on synthetic data, then the exact `segpt-train` command line for your own data.
4. (Optional) FP32 ONNX export.

In [1]:
# setup — all imports live here; CPU-only, seeded, synthetic data throughout
import tempfile, warnings
import numpy as np
import torch
from omegaconf import OmegaConf
from torch.utils.data import Dataset, DataLoader

from segmentation_tools_pt.model import (
    get_model_from_cfg, get_param_groups, freeze_encoder, unfreeze_encoder)
from segmentation_tools_pt.hydra_pipeline import resolve_train_schedule
from segmentation_tools_pt.train import train

torch.manual_seed(0)
np.random.seed(0)

## 1. One factory, two model sources

`get_model_from_cfg(cfg, in_channels, crop_size)` reads `cfg.model.source`. `crop_size` is passed explicitly
(it comes from `train.crop_size`) so the factory can refuse a crop that is too small for the chosen depth.

In [2]:
base = {'train': {'num_classes': 1, 'crop_size': 64, 'freeze_epochs': 5, 'encoder_lr': 1e-4, 'decoder_lr': 1e-3}}
cfg_smp = OmegaConf.create({**base, 'model': {'source': 'smp', 'architecture': 'unet', 'arch': 'mobilenet_v2', 'pretrained': False}})
cfg_cus = OmegaConf.create({**base, 'model': {'source': 'custom_unet', 'custom_unet': {'encoder_depth': 3, 'base_channels': 16}}})

x = torch.randn(2, 1, 64, 64)
for name, cfg in [('smp (mobilenet_v2 UNet)', cfg_smp), ('custom_unet (depth 3)', cfg_cus)]:
    m = get_model_from_cfg(cfg, in_channels=1, crop_size=64).eval()
    with torch.no_grad():
        y = m(x)
    n = sum(p.numel() for p in m.parameters())
    print(f'{name:26s} params={n:>10,}  output={tuple(y.shape)}')
    assert y.shape == (2, 1, 64, 64)

smp (mobilenet_v2 UNet)    params= 6,628,369  output=(2, 1, 64, 64)
custom_unet (depth 3)      params=   482,449  output=(2, 1, 64, 64)


Both models answer the same question — raw logits of shape `(B, num_classes, H, W)` — so `train()`, AMP and ONNX export
do not care which one you picked. The custom model also exposes `.encoder`, `.decoder` and `.segmentation_head`, so the
helpers that give the encoder and decoder different learning rates keep working:

In [3]:
m = get_model_from_cfg(cfg_cus, in_channels=1, crop_size=64)
groups = get_param_groups(m, encoder_lr=1e-4, decoder_lr=1e-3)
print([(len(g['params']), g['lr']) for g in groups])
freeze_encoder(m);   assert not any(p.requires_grad for p in m.encoder.parameters())
unfreeze_encoder(m); assert all(p.requires_grad for p in m.encoder.parameters())

[(24, 0.0001), (26, 0.001)]


Guard rails fail **before** a model is built, with the fix in the message. Options that arrive in later tickets
(residual blocks, ASPP/attention) say so instead of silently doing something else:

In [4]:
def attempt(**custom):
    cfg = OmegaConf.create({**base, 'model': {'source': 'custom_unet', 'custom_unet': custom}})
    try:
        get_model_from_cfg(cfg, in_channels=1, crop_size=64)
    except (ValueError, NotImplementedError) as e:
        print(f'{type(e).__name__}: {e}')

attempt(block_style='residual')      # not yet (#63)
attempt(encoder_depth='4')           # a string from the CLI, not an integer
attempt(encoder_depth=7)             # 64 px cannot be halved 7 times
attempt(norm='layer')                # unknown norm

NotImplementedError: custom_unet.block_style='residual' is not implemented yet; only 'double'
ValueError: custom_unet.encoder_depth must be a positive integer, 'recommend' or 'auto', got '4'
ValueError: crop_size=64 is too small for encoder_depth=7 (needs >= 128)
ValueError: custom_unet.norm='layer' is not supported; choose one of ['group', 'batch', 'instance']


## 1b. Let the crop pick the depth, and choose the norm (#62)

`encoder_depth: recommend` or `auto` sizes the net from `train.crop_size`: the bottleneck should end up near
`bottleneck_min_size` (default 32) pixels per side. `auto` takes `floor(log2(crop / min))`, `recommend` rounds to the nearest
step; both are clamped to 3..6 downsamplings. An integer is used exactly as written. `norm` picks `group` (default),
`batch` or `instance`.

In [5]:
from segmentation_tools_pt.model import resolve_encoder_depth
for crop in (128, 384, 512, 1024):
    print(crop, 'recommend ->', resolve_encoder_depth('recommend', crop), '| auto ->', resolve_encoder_depth('auto', crop))
assert resolve_encoder_depth('recommend', 512) == 4 and resolve_encoder_depth(2, 512) == 2
for norm in ('group', 'batch', 'instance'):
    c = OmegaConf.create({**base, 'model': {'source': 'custom_unet', 'custom_unet': {'encoder_depth': 'recommend', 'norm': norm, 'base_channels': 8}}})
    net = get_model_from_cfg(c, in_channels=1, crop_size=64).eval()
    print(norm, 'depth', len(net.encoder.stages) - 1, tuple(net(torch.randn(1, 1, 64, 64)).shape))

128 recommend -> 3 | auto -> 3
384 recommend -> 4 | auto -> 3
512 recommend -> 4 | auto -> 4
1024 recommend -> 5 | auto -> 5
group depth 3 (1, 1, 64, 64)
batch depth 3 (1, 1, 64, 64)
instance depth 3 (1, 1, 64, 64)


## 2. Random weights change the training schedule

An SMP encoder usually starts from ImageNet weights, so the default recipe **freezes it for a few epochs** and later trains
it at a **10× lower learning rate** than the decoder. A custom UNet starts from random weights: a frozen random encoder
just feeds noise to the decoder. `resolve_train_schedule(cfg)` therefore returns `freeze_epochs=0` and
`encoder_lr = decoder_lr` for `custom_unet`, with a warning, and leaves `smp` values untouched.

In [6]:
print('smp         ->', resolve_train_schedule(cfg_smp))
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    sched = resolve_train_schedule(cfg_cus)
print('custom_unet ->', sched)
print('warning     :', str(w[0].message)[:140], '...')
assert sched == (0, 1e-3, 1e-3)

# explicit override (e.g. a later warm-start): keys live under model.custom_unet
cfg_keep = OmegaConf.create({**base, 'model': {'source': 'custom_unet', 'custom_unet': {'freeze_epochs': 2, 'encoder_lr': 5e-4}}})
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print('override    ->', resolve_train_schedule(cfg_keep))

smp         -> TrainSchedule(freeze_epochs=5, encoder_lr=0.0001, decoder_lr=0.001)
custom_unet -> TrainSchedule(freeze_epochs=0, encoder_lr=0.001, decoder_lr=0.001)
override    -> TrainSchedule(freeze_epochs=2, encoder_lr=0.0005, decoder_lr=0.001)


## 3. Train with the library

Synthetic task: a 1-channel 64×64 image with a few bright squares on noise; the mask is the squares. Small enough for CPU
in seconds. `train()` is called exactly as `segpt-train` calls it — the only model-specific parts are the factory call and
the resolved schedule.

In [7]:
H = W = 64
class Squares(Dataset):
    def __init__(self, n, seed): self.n, self.seed = n, seed
    def __len__(self): return self.n
    def __getitem__(self, i):
        g = np.random.default_rng(self.seed * 10_000 + i)
        mask = np.zeros((H, W), np.float32)
        for _ in range(g.integers(1, 4)):
            s = g.integers(8, 16); y, x = g.integers(0, H - s), g.integers(0, W - s); mask[y:y + s, x:x + s] = 1
        img = (mask * 1.0 + g.normal(0, 0.5, (H, W))).astype(np.float32)[None]
        return {'image': torch.from_numpy(img), 'mask': torch.from_numpy(mask[None]), 'sample_id': str(i)}

train_loader = DataLoader(Squares(96, 0), batch_size=16, shuffle=True, generator=torch.Generator().manual_seed(0))
val_loader = DataLoader(Squares(32, 1), batch_size=16)

model = get_model_from_cfg(cfg_cus, in_channels=1, crop_size=H)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    freeze, enc_lr, dec_lr = resolve_train_schedule(cfg_cus)
with tempfile.TemporaryDirectory() as d:
    hist = train(root=None, model=model, train_loader=train_loader, val_loader=val_loader, num_classes=1, num_epochs=12,
                 batch_size=16, freeze_epochs=freeze, encoder_lr=enc_lr, decoder_lr=dec_lr, crop_size=H,
                 device_str='cpu', checkpoint_dir=d, use_amp=False, resume=False, task='binary', seed=0, num_workers=0)
first, last = hist[0], hist[-1]
print(f"val IoU: epoch 1 = {first['val_iou']:.3f}  ->  epoch {len(hist)} = {last['val_iou']:.3f}")
assert last['val_iou'] > first['val_iou'], 'custom UNet did not learn on the synthetic task'
assert last['val_iou'] > 0.5

/home/hasan-spark/workspace/projects/git_data/seg_pt_wt_62/src/segmentation_tools_pt/train.py:529: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:822.)
  state.loss_dict = {k: float(v) for k, v in ld.items()}


epoch 001/12  train_loss=0.8214  val_loss=0.7789  train_iou=0.1331  val_iou=0.4477  val_dice=0.6185  val_recall=0.9989  lr=9.83e-04  *best*


epoch 002/12  train_loss=0.7169  val_loss=0.7333  train_iou=0.6445  val_iou=0.6209  val_dice=0.7661  val_recall=0.9987  lr=9.33e-04  *best*


epoch 003/12  train_loss=0.6807  val_loss=0.7139  train_iou=0.6864  val_iou=0.6656  val_dice=0.7993  val_recall=0.9980  lr=8.54e-04  *best*


epoch 004/12  train_loss=0.6641  val_loss=0.7020  train_iou=0.7384  val_iou=0.6770  val_dice=0.8074  val_recall=0.9988  lr=7.50e-04  *best*


epoch 005/12  train_loss=0.6526  val_loss=0.6928  train_iou=0.7295  val_iou=0.7106  val_dice=0.8308  val_recall=0.9977  lr=6.30e-04  *best*


... (4 outputs left out)

epoch 010/12  train_loss=0.6224  val_loss=0.6686  train_iou=0.7818  val_iou=0.7515  val_dice=0.8581  val_recall=0.9976  lr=6.79e-05  *best*


epoch 011/12  train_loss=0.6208  val_loss=0.6676  train_iou=0.7899  val_iou=0.7521  val_dice=0.8585  val_recall=0.9976  lr=1.80e-05  *best*


epoch 012/12  train_loss=0.6199  val_loss=0.6674  train_iou=0.7896  val_iou=0.7521  val_dice=0.8585  val_recall=0.9976  lr=1.00e-06


val IoU: epoch 1 = 0.448  ->  epoch 12 = 0.752


Now the same thing on **your** data. The YAML/CLI route needs no code:

```bash
# patch-cache build is unchanged; then train the custom UNet
segpt-train configs/train.yaml \
    model.source=custom_unet \
    +model.custom_unet.encoder_depth=4 \
    model.weights_path=null model.pretrained=false \
    train.crop_size=512
```

- Leave `source` out (or `source=smp`) for today's behaviour — nothing else changes.
- `model.weights_path=null model.pretrained=false` silences the "random-init, pretrained ignored" warning.
- `encoder_depth` must satisfy `train.crop_size >= 2**encoder_depth`; for 512 px, depth 4 gives a 32 px bottleneck.
- To keep a pretrained-style schedule anyway, set `model.custom_unet.freeze_epochs` / `model.custom_unet.encoder_lr`.

## 4. Optional: FP32 ONNX export

Skipped automatically when `onnx` / `onnxruntime` are not installed (they are optional dependencies).

In [8]:
try:
    import onnx, onnxruntime  # noqa: F401
    from segmentation_tools_pt.export import export_onnx
    with tempfile.TemporaryDirectory() as d:
        path = export_onnx(model.cpu(), f'{d}/custom_unet.onnx', in_channels=1, input_size=H, task='binary', verify=True)
        print('exported + verified against PyTorch:', path.split('/')[-1])
except ImportError as e:
    print('skipped (optional dependency missing):', e)

skipped (optional dependency missing): No module named 'onnx'


## Where next

- Design and decisions: `docs/superpowers/specs/2026-10-01-custom-unet-smp-factory-design.md` (map #12).
- Coming in follow-up tickets: residual blocks (#63), ASPP (#64), attention + ONNX check (#65), docs (#66).
- The random-init gap versus DINOv3 / pretrained SMP encoders is real: for domain shift, pretrained encoders remain the stronger choice until warm-start support exists.
- Ask the agent: the `choose-model-architecture` skill now covers when to pick `custom_unet`.